# Assistant biomédical à verdict calibré — Jour 3
### Calibration, seuil d'abstention, détection hors périmètre, index FAISS

Ce notebook suppose que le **Jour 2** a été exécuté : adaptateur LoRA sauvegardé,
splits `final_train.json` / `final_val.json` / `final_test_expert_holdout.json`
disponibles sur Drive.

Objectifs du jour :
1. Calibrer le modèle (température scaling) et mesurer l'ECE avant/après
2. Choisir un seuil d'abstention justifié empiriquement
3. Construire une détection des questions hors périmètre biomédical
4. Construire l'index FAISS pour citer la phrase source qui justifie le verdict


## 1. Setup

In [ ]:
!pip install -q transformers peft sentence-transformers faiss-cpu scikit-learn


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJECT_DIR = "/content/drive/MyDrive/assistant_biomedical"
DATA_DIR = f"{PROJECT_DIR}/data"
CKPT_DIR = f"{PROJECT_DIR}/checkpoints"
RESULTS_DIR = f"{PROJECT_DIR}/results"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device :", device)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


## 2. Rechargement du modèle fine-tuné (base + adaptateur LoRA)

In [ ]:
MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
ADAPTER_DIR = f"{CKPT_DIR}/pubmedbert_lora_adapter_final"

LABEL2ID = {"yes": 0, "no": 1, "maybe": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
MAX_LENGTH = 384

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.to(device)
model.eval()
print("Modèle fine-tuné rechargé.")


## 3. Chargement des splits (validation pour calibrer, test held-out pour évaluer)

In [ ]:
final_val = load_json(f"{DATA_DIR}/final_val.json")
final_test = load_json(f"{DATA_DIR}/final_test_expert_holdout.json")

print("Validation :", len(final_val))
print("Test held-out :", len(final_test))


## 4. Extraction des logits bruts

On calcule les logits (avant softmax) pour la validation et pour le test.
La calibration s'ajuste **uniquement sur la validation**, jamais sur le test,
pour éviter toute fuite d'information.

In [ ]:
@torch.no_grad()
def get_logits(data, batch_size=32):
    all_logits = []
    all_labels = []
    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]
        questions = [ex["question"] for ex in batch]
        contexts = [ex["context"] for ex in batch]
        labels = [LABEL2ID[ex["label"]] for ex in batch]

        inputs = tokenizer(
            questions, contexts,
            truncation=True, max_length=MAX_LENGTH,
            padding=True, return_tensors="pt"
        ).to(device)

        outputs = model(**inputs)
        all_logits.append(outputs.logits.cpu())
        all_labels.extend(labels)

    return torch.cat(all_logits, dim=0), torch.tensor(all_labels)

val_logits, val_labels = get_logits(final_val)
test_logits, test_labels = get_logits(final_test)

print("Logits validation :", val_logits.shape)
print("Logits test       :", test_logits.shape)


## 5. Fonction ECE (Expected Calibration Error)

In [ ]:
def compute_ece(probs, labels, n_bins=15):
    confidences, predictions = torch.max(probs, dim=1)
    accuracies = predictions.eq(labels)

    ece = torch.zeros(1)
    bin_boundaries = torch.linspace(0, 1, n_bins + 1)

    for i in range(n_bins):
        lo, hi = bin_boundaries[i], bin_boundaries[i+1]
        in_bin = (confidences > lo) & (confidences <= hi)
        prop_in_bin = in_bin.float().mean()
        if prop_in_bin.item() > 0:
            acc_in_bin = accuracies[in_bin].float().mean()
            conf_in_bin = confidences[in_bin].mean()
            ece += torch.abs(conf_in_bin - acc_in_bin) * prop_in_bin

    return ece.item()

# ECE avant calibration (température = 1, softmax brut)
raw_probs_test = F.softmax(test_logits, dim=1)
ece_before = compute_ece(raw_probs_test, test_labels)
print(f"ECE avant calibration (test held-out) : {ece_before:.4f}")


## 6. Température scaling

On apprend un unique paramètre scalaire T qui divise les logits avant le softmax,
optimisé pour minimiser la NLL sur la validation. Le modèle reste gelé : seul T
est appris.

In [ ]:
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature

temp_scaler = TemperatureScaler()

nll_criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.LBFGS([temp_scaler.temperature], lr=0.01, max_iter=100)

def eval_closure():
    optimizer.zero_grad()
    loss = nll_criterion(temp_scaler(val_logits), val_labels)
    loss.backward()
    return loss

optimizer.step(eval_closure)

learned_temperature = temp_scaler.temperature.item()
print(f"Température apprise : {learned_temperature:.4f}")


In [ ]:
# ECE après calibration, sur le test held-out
calibrated_logits_test = test_logits / learned_temperature
calibrated_probs_test = F.softmax(calibrated_logits_test, dim=1)
ece_after = compute_ece(calibrated_probs_test, test_labels)

print(f"ECE avant calibration : {ece_before:.4f}")
print(f"ECE après calibration : {ece_after:.4f}")

save_json(
    {"temperature": learned_temperature, "ece_before": ece_before, "ece_after": ece_after},
    f"{RESULTS_DIR}/day3_calibration.json"
)


## 7. Choix du seuil d'abstention

On balaie plusieurs seuils de confiance (probabilité calibrée max) sur la
**validation** et on observe le compromis accuracy / taux de couverture
(proportion de cas où le modèle répond au lieu de s'abstenir).

In [ ]:
calibrated_logits_val = val_logits / learned_temperature
calibrated_probs_val = F.softmax(calibrated_logits_val, dim=1)
val_confidences, val_predictions = torch.max(calibrated_probs_val, dim=1)

thresholds = [0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8]
print(f"{'Seuil':>6s} {'Couverture':>12s} {'Accuracy (répondus)':>22s}")
for t in thresholds:
    mask = val_confidences >= t
    coverage = mask.float().mean().item()
    if mask.sum() > 0:
        acc = val_predictions[mask].eq(val_labels[mask]).float().mean().item()
    else:
        acc = float("nan")
    print(f"{t:>6.2f} {coverage:>12.2%} {acc:>22.4f}")


**Choix retenu : seuil = 0,6** (valeur du cahier des charges), à ajuster si le
tableau ci-dessus montre un compromis nettement meilleur pour ton cas d'usage.

In [ ]:
ABSTENTION_THRESHOLD = 0.6

calibrated_logits_test = test_logits / learned_temperature
calibrated_probs_test = F.softmax(calibrated_logits_test, dim=1)
test_confidences, test_predictions = torch.max(calibrated_probs_test, dim=1)

abstain_mask = test_confidences < ABSTENTION_THRESHOLD
coverage_test = (~abstain_mask).float().mean().item()

answered_acc = test_predictions[~abstain_mask].eq(test_labels[~abstain_mask]).float().mean().item()
overall_acc_no_abstention = test_predictions.eq(test_labels).float().mean().item()

print(f"Seuil d'abstention retenu : {ABSTENTION_THRESHOLD}")
print(f"Taux de couverture (test held-out) : {coverage_test:.2%}")
print(f"Accuracy sur les cas répondus : {answered_acc:.4f}")
print(f"Accuracy globale sans abstention (référence) : {overall_acc_no_abstention:.4f}")

save_json({
    "threshold": ABSTENTION_THRESHOLD,
    "coverage": coverage_test,
    "accuracy_on_answered": answered_acc,
    "accuracy_no_abstention": overall_acc_no_abstention,
}, f"{RESULTS_DIR}/day3_abstention.json")


## 8. Index FAISS — deux usages

1. **Citation du passage source** : pour une paire (question, contexte) donnée, on
   découpe le contexte en phrases et on retrouve la phrase la plus proche de la
   question (similarité cosinus) → c'est la phrase citée comme preuve.
2. **Détection hors périmètre** : on indexe globalement les phrases du corpus
   d'entraînement biomédical. Une question dont le plus proche voisin est très
   éloigné (similarité faible) est probablement hors du périmètre biomédical.


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import re

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

def split_sentences(text):
    # Découpage simple par ponctuation forte, suffisant pour des abstracts PubMed
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if len(s.strip()) > 5]


In [ ]:
def cite_evidence_sentence(question, context, top_k=1):
    sentences = split_sentences(context)
    if not sentences:
        return [context]

    sentence_embeddings = embedder.encode(sentences, convert_to_tensor=True, normalize_embeddings=True)
    question_embedding = embedder.encode([question], convert_to_tensor=True, normalize_embeddings=True)

    similarities = (sentence_embeddings @ question_embedding.T).squeeze(1)
    top_indices = torch.topk(similarities, k=min(top_k, len(sentences))).indices.tolist()

    return [sentences[i] for i in top_indices]

# Test rapide
example = final_test[0]
cited = cite_evidence_sentence(example["question"], example["context"])
print("Question :", example["question"])
print("Phrase citée comme preuve :", cited[0])


### Index global pour la détection hors périmètre

On indexe toutes les phrases des contextes d'entraînement (`final_train`).

In [ ]:
final_train = load_json(f"{DATA_DIR}/final_train.json")

all_train_sentences = []
for ex in final_train:
    all_train_sentences.extend(split_sentences(ex["context"]))

# On limite à un sous-échantillon si trop volumineux, pour rester raisonnable sur T4
MAX_INDEX_SENTENCES = 60000
if len(all_train_sentences) > MAX_INDEX_SENTENCES:
    random.shuffle(all_train_sentences)
    all_train_sentences = all_train_sentences[:MAX_INDEX_SENTENCES]

print("Nombre de phrases indexées :", len(all_train_sentences))

corpus_embeddings = embedder.encode(
    all_train_sentences, batch_size=128, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
)

dimension = corpus_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)  # produit scalaire = cosinus (embeddings normalisés)
faiss_index.add(corpus_embeddings)

faiss.write_index(faiss_index, f"{CKPT_DIR}/biomedical_corpus.index")
save_json(all_train_sentences, f"{DATA_DIR}/faiss_corpus_sentences.json")
print("Index FAISS construit et sauvegardé.")


## 9. Détection hors périmètre biomédical

Le dataset PubMedQA ne contient que des questions biomédicales : il n'y a donc pas
d'exemples "hors périmètre" à disposition. On construit un petit jeu synthétique
de questions clairement non biomédicales pour tester le détecteur.

In [ ]:
synthetic_out_of_scope = [
    "Quelle est la capitale de la France ?",
    "Comment faire une bonne pâte à crêpes ?",
    "Qui a gagné la coupe du monde de football en 2018 ?",
    "Quel est le meilleur langage de programmation pour débuter ?",
    "Explique-moi les règles du jeu d'échecs.",
    "Quelle est la météo prévue pour demain ?",
    "Recommande-moi un bon roman de science-fiction.",
    "Comment réparer une fuite d'eau sous l'évier ?",
    "Quels sont les meilleurs endroits à visiter à Abidjan ?",
    "Comment composer une chanson de rap ?",
]

def ood_score(question, k=5):
    """Score d'éloignement : 1 - similarité moyenne aux k plus proches voisins.
    Plus le score est élevé, plus la question est probablement hors périmètre."""
    q_emb = embedder.encode([question], convert_to_numpy=True, normalize_embeddings=True)
    similarities, _ = faiss_index.search(q_emb, k)
    mean_similarity = similarities.mean()
    return 1 - mean_similarity

# Scores pour des questions biomédicales du test (in-domain)
in_domain_scores = [ood_score(ex["question"]) for ex in final_test[:30]]

# Scores pour les questions synthétiques hors périmètre
ood_scores = [ood_score(q) for q in synthetic_out_of_scope]

print("Score moyen d'éloignement — questions biomédicales (in-domain) :", np.mean(in_domain_scores))
print("Score moyen d'éloignement — questions hors périmètre (synthétique) :", np.mean(ood_scores))


In [ ]:
# Choix d'un seuil de détection hors périmètre, à mi-chemin entre les deux distributions
OOS_THRESHOLD = (np.mean(in_domain_scores) + np.mean(ood_scores)) / 2
print(f"Seuil de détection hors périmètre retenu : {OOS_THRESHOLD:.4f}")

correctly_flagged = sum(1 for s in ood_scores if s > OOS_THRESHOLD)
false_alarms = sum(1 for s in in_domain_scores if s > OOS_THRESHOLD)

print(f"Questions hors périmètre correctement détectées : {correctly_flagged}/{len(ood_scores)}")
print(f"Fausses alertes sur questions in-domain : {false_alarms}/{len(in_domain_scores)}")

save_json({
    "oos_threshold": float(OOS_THRESHOLD),
    "mean_in_domain_score": float(np.mean(in_domain_scores)),
    "mean_ood_score": float(np.mean(ood_scores)),
    "correctly_flagged": correctly_flagged,
    "false_alarms": false_alarms,
}, f"{RESULTS_DIR}/day3_ood_detection.json")


## 10. Pipeline de démonstration complet (verdict + confiance + abstention + citation + garde-fou hors périmètre)

In [ ]:
@torch.no_grad()
def full_pipeline(question, context):
    # 1. Détection hors périmètre
    score = ood_score(question)
    if score > OOS_THRESHOLD:
        return {
            "verdict": None,
            "message": "Question hors du périmètre biomédical couvert par cet assistant.",
        }

    # 2. Prédiction + calibration
    inputs = tokenizer(question, context, truncation=True, max_length=MAX_LENGTH,
                        padding=True, return_tensors="pt").to(device)
    logits = model(**inputs).logits.cpu()
    probs = F.softmax(logits / learned_temperature, dim=1).squeeze(0)
    confidence, pred_id = torch.max(probs, dim=0)
    confidence, pred_id = confidence.item(), pred_id.item()

    # 3. Abstention
    if confidence < ABSTENTION_THRESHOLD:
        verdict = "incertain"
    else:
        verdict = ID2LABEL[pred_id]

    # 4. Citation de la phrase source
    citation = cite_evidence_sentence(question, context)[0]

    return {
        "verdict": verdict,
        "raw_label": ID2LABEL[pred_id],
        "confidence": round(confidence, 4),
        "citation": citation,
        "message": "Cet outil ne fournit pas de conseil médical ni de diagnostic.",
    }

# Démonstration sur quelques exemples du test held-out + une question hors périmètre
for ex in final_test[:3]:
    result = full_pipeline(ex["question"], ex["context"])
    print("Question :", ex["question"])
    print("Vrai label :", ex["label"])
    print("Résultat pipeline :", result)
    print("---")

print("=== Test hors périmètre ===")
print(full_pipeline("Quelle est la capitale de la France ?", ""))


## 11. Bilan du Jour 3

- ✅ Température scaling appris sur la validation (T = valeur imprimée ci-dessus), ECE avant/après mesuré sur le test held-out
- ✅ Seuil d'abstention choisi et justifié par le compromis accuracy/couverture
- ✅ Index FAISS construit (corpus + citation par phrase au sein du contexte)
- ✅ Détecteur hors périmètre biomédical basé sur la distance FAISS, testé sur des questions synthétiques
- ✅ Pipeline de démonstration complet fonctionnel (verdict + confiance + abstention + citation + garde-fou)

**Prochaine étape (Jour 4)** : intégration du LLM génératif (Qwen2.5-1.5B-Instruct)
pour transformer le verdict + la phrase citée en explication en langage naturel,
tests supplémentaires et analyse d'erreurs.